# Chapter 2. Patterns for Cleaner Python

## 2.3 Context Managers and the with Statement

with: effectively manage the resources


### with file: 

In [1]:
with open('hello.txt', 'w') as f:
    f.write('hello World!')
    

In [3]:
f = open('hello.txt', 'w')
try:
    f.write('hello World again!')
finally:
    f.close()

### with threading lock class: 

Threading: allows running multiple threads concurrently in one single process.

In [5]:
import threading
import time

def crawl(link, delay=3):
    print(f"crawl started for {link}")
    time.sleep(delay)  # Blocking I/O (simulating a network request)
    print(f"crawl ended for {link}")

links = [
    "https://python.org",
    "https://docs.python.org",
    "https://peps.python.org",
]

# Start threads for each link
threads = []
for link in links:
    # Using `args` to pass positional arguments and `kwargs` for keyword arguments
    t = threading.Thread(target=crawl, args=(link,), kwargs={"delay": 2})
    threads.append(t)

# Start each thread
for t in threads:
    t.start()

# Wait for all threads to finish
for t in threads:
    t.join()

crawl started for https://python.org
crawl started for https://docs.python.org
crawl started for https://peps.python.org
crawl ended for https://python.orgcrawl ended for https://docs.python.org
crawl ended for https://peps.python.org



### with own objects: implement Context Managers. 


Just need to create \__enter__ and \__exit__ method.

## 2.4 Underscores, Dunders, and More

Python code style guide: https://peps.python.org/pep-0008/

Dunders: two underscore characters "__". 

Meanings:

- Single leading underscore (prefix, eg: "_var"): by convention it means for internal use. A hint for programmers but not enforced by the interpreter (except in wildcard imports).
- Single trailing underscore (postfix, eg: "var_"): used by convention to avoid naming conflicts with Python keywords.
- Double leading underscores (prefix, eg: "__var"): triggers name mangling when used in a class context. Enforced by the Python interpreter.
- Double leading and trailing underscores (eg: "\__get__"): indicate special methods defined by the Python language. Avoid this naming scheme for your own attributes.
- Single underscore "_":
    - a name for temporary or insignificant variable
    - represents the result of the last expression in a Python REPL session.

In [29]:
class Test:
    def __init__(self):
        self.foo=11
        self._bas = 12
        self.__bas = 13
        self.bas_=14

class ExtendedTest(Test):
    def __init__(self):
        super().__init__()
        self.foo='overwritten'
        self._bas = 'overwritten'
        self.__bas = 'overwritten'

    def get_mangled(self):
        return self.__bas

In [24]:
t = Test()
dir(t)

['_Test__bas',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_bas',
 'bas_',
 'foo']

In [25]:
t2=ExtendedTest()
dir(t2)

['_ExtendedTest__bas',
 '_Test__bas',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_bas',
 'bas_',
 'foo',
 'get_mangled']

In [26]:
t2._ExtendedTest__bas

'overwritten'

In [27]:
t2._Test__bas

13

In [28]:
t2.get_mangled()

'overwritten'

In [32]:
_MangledGlobal__mangled=23

class MangledGlobal:
    def test(self):
        return __mangled

In [33]:
MangledGlobal().test()

23

In [39]:
_yetAnother__example = 'qinqin'

class yetAnother:
    def surprise(self):
        return __example
    def get(self):
        return __Test

In [40]:
yetAnother().surprise()
yetAnother().get()

NameError: name '_yetAnother__Test' is not defined

In [41]:
_

'qinqin'

## 2.5 String Formatting 
Dan's Python String Formatting Rule of Thumb:
- If your format strings are user-supplied, use Template Strings to avoid security issues;
- Else, if you're on Python 3.6+, use Literal String Interpolation;
- Else, use "New Style" String Formatting. 

### "Old Style" string formatting: with %

In [52]:
name = 'Qin'
'Hello, %s' % name

'Hello, Qin'

In [47]:
errno=50159747054
"%x" % errno

'badc0ffee'

In [50]:
# make multiple substitutions
"Hey %s, there is an 0x%x error!" % (name, errno)

'Hey Qin, there is an 0xbadc0ffee error!'

In [51]:
# refer to variable substition by name => so don't need to worry about order!
"Hey %(name)s, there is an 0x%(errorNumber)x error!" % {"name":name, "errorNumber": errno}

'Hey Qin, there is an 0xbadc0ffee error!'

### "New Style" string formatting: with .format()

In [53]:
"Hello, {}".format(name)

'Hello, Qin'

In [57]:
"Hey {name}, there is an 0x{error:x} error!".format(name=name, error=errno)

'Hey Qin, there is an 0xbadc0ffee error!'

### Literal String Interpolation

Started in Python 3.6, allowing using embedded Python expressions, and we can even do inline arithmetic with it. 

In [60]:
f"Hello, {name}"

'Hello, Qin'

In [61]:
# inline arithmetic

a = 10
b = 5
f"Five plus ten is {a+b}"

'Five plus ten is 15'

In [62]:
# support showing the string in different format 
f"Hey {name}, there is an 0x{errno:x} error!"

'Hey Qin, there is an 0xbadc0ffee error!'

#### Extension: difference between f string and r string

r prefix: raw string literals, the main differene is backlash \ is really taken as a backlash. In f string, to show a backlash we need to double it: \\. 
https://stackoverflow.com/questions/2081640/what-exactly-do-u-and-r-string-prefixes-do-and-what-are-raw-string-literals

In [73]:
f"Hey {name}, there is an 0x{errno:x} error!"

'Hey Qin, there is an 0xbadc0ffee error!'

In [74]:
r"Hey {name}, there is an 0x{errno:x} error!"

'Hey {name}, there is an 0x{errno:x} error!'

In [79]:
print(r"here is what looks like when I have another \'quote\' in r string")
print(f"here is what looks like when I have another \'quote\' in f string")
print(f"here is what looks like when I have another \\'quote\\' in f string")
print(u"here is what looks like when I have another \'quote\' in f string")

here is what looks like when I have another \'quote\' in r string
here is what looks like when I have another 'quote' in f string
here is what looks like when I have another \'quote\' in f string
here is what looks like when I have another 'quote' in f string


### Template Strings

Less used, but could be useful in certain context. 

Note: Template string doesn't allow format specification!!

Template string can be used when handling format strings generated by users of the program. 

In [82]:
from string import Template
t = Template('Hey, $name!')
t.substitute(name = name)

'Hey, Qin!'

In [84]:
t = Template("Hey $name, there is an 0x$error error!")
t.substitute(name = name, error=errno)

'Hey Qin, there is an 0x50159747054 error!'

In [85]:
t.substitute(name = name, error = hex(errno))

'Hey Qin, there is an 0x0xbadc0ffee error!'

In [86]:
import this

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!


# Chapter 3. Effective Functions 

All data in a Python program is represented by objects or relations between objects. Strings, lists, modules and functions are all objects. 

## 3.1 Python's Functions Are First-Class objects

You can:
- assign them to variables
- store them in data structures
- pass them as arguments to other functions
- return them as values from other functions.

### Functions are objects

In [88]:
def yell(text):
    return text.upper() + "!"
yell("hello")

'HELLO!'

In [89]:
# assign function to another variable
bark = yell
bark('hey')

'HEY!'

In [90]:
# Delete the function name
del yell

In [91]:
# yet function still exists
bark('yes')

'YES!'

In [94]:
# Python attaches a string identifier to every function at create time for debugging purpose
bark.__name__

'yell'

In [95]:
yell.__name__

NameError: name 'yell' is not defined

In [96]:
?bark

Signature: bark(text)
Docstring: <no docstring>
File:      /var/folders/_p/cpzbk4z16cj_v913_1nkmx200000gn/T/ipykernel_13348/3215279926.py
Type:      function

### Functions can be stored in Data Structures

In [101]:
# Functions can be saved in a data structure
f_list = [bark, str.lower, str.capitalize]
f_list

[<function __main__.yell(text)>,
 <method 'lower' of 'str' objects>,
 <method 'capitalize' of 'str' objects>]

### Functions can be passed to other functions

In [104]:
# Functions can be passed to Other Functions
def greet(func):
    greeting = func(f"Nice to meet you")
    print(greeting)

greet(bark)

NICE TO MEET YOU!


Functions that can accept other functions as arguments are also called __higher order functions__. It allows us to abstract away and pass around _behaviours_ in the programs. Classical example: map function.

In [108]:
# example of map function
list(map(str.upper, ['hello', "I", "am"]))

['HELLO', 'I', 'AM']

In [109]:
list(map(bark, ['hello', "I", "am"]))

['HELLO!', 'I!', 'AM!']

### Functions can be nested and can also capture local state

Functions can be nested, and nested functions can CAPTURE and CARRY some of the parent function's state. This is called __lexical closues__: a closure remembers the values from its enclosing lexical scope even when the program flow is no longer in that scope. 

This means, not only functions can _return behaviours_, but they can also _pre-configure those behaviours_. 

In [117]:
# Example of nested functions

def make_adder(n):
    def add(x):
        return x+n
    return add

plus_4 = make_adder(4)
plus_5 = make_adder(5)

In [118]:
plus_4(2.3)

6.3

In [119]:
plus_5(1.4)

6.4

In [120]:
make_adder(10)(3.4)

13.4

### Objects can behave like functions 

The non-functions objects can be (made) _callable_, which means you can use the round parentheses function call syntax on it and even pass in function call arguments. For class this can be defined by \__call__ dunder method.

There is a built-in function "callable" to check whether an object is callable. 

In [122]:
# define a callable object

class Adder:
    def __init__(self, n):
        self.n = n

    def __call__(self, x):
        return self.n + x

plus_4 = Adder(4)
plus_4(5)

9

In [123]:
dir(plus_4)

['__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'n']

In [124]:
callable(plus_4)

True

In [125]:
callable(bark)

True

In [127]:
callable(name)

False

## 3.2 Lambdas Are Single-Expression Functions

Difference between lambda function and "def" functions:
- no need to assign a name to lambda function!
- lambda functions are restricted to a single expression.

HOWEVER, lambda functions should be used sparingly and with extraodinary care. 

If you find yourself doing anything remotely complex with lambda expressions, consider defining a standalone function with a proper name instead. 

In [129]:
(lambda x,y:x+y)(2, 4)

6

In [131]:
# lambda function also work as lexical closures
def make_adder(n):
    return lambda x: x+n

In [132]:
make_adder(3)(4.2)

7.2

## 3.3 The Power of Decorators 

A decorator is a _callable_ that takes a _callable_ as input and returns another _callable_.
Decorators allow you to define reusable building blocks that can change or extend the behaviour of other functions. 

### Decorator basics 

In [144]:
def uppercase(func):
    def wrapper():
        output = func()
        output = output.upper()
        return output
    return wrapper

@uppercase
def greet():
    return "hello"



In [145]:
greet()

'HELLO'

In [146]:
greet

<function __main__.uppercase.<locals>.wrapper()>

In [147]:
uppercase(greet)

<function __main__.uppercase.<locals>.wrapper()>

Extension: most useful decorators for functions and classes?

### Decorator stacking: apply multiple decorators to a function

In [148]:
# Apply multiple decorators to a function: nearest is applied first => Decorator stacking

def strong(func):
    def wrapper():
        return '<strong>' + func() + '</strong>'
    return wrapper

def emphasis(func):
    def wrapper():
        return '<em>' + func() + '</em>'
    return wrapper

    

In [149]:
@emphasis
@strong
def greet():
    return 'hello'

In [150]:
greet()

'<em><strong>hello</strong></em>'

### Decorating Functions That Accepts Arguments

In [154]:
def proxy(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def trace(func):
    def wrapper(*args, **kwargs):
        print(f'TRACE: calling {func.__name__}() '
              f'with {args}, {kwargs}')
        original_result = func(*args, **kwargs)
        print(f'TRACE: {func.__name__}() '
              f'returned {original_result}')
        return original_result
    return wrapper

In [155]:
@trace
def say(name, line):
    return f'{name}: {line}'

say('Gaby', 'Great')

TRACE: calling say() with ('Gaby', 'Great'), {}
TRACE: say() returned Gaby: Great


'Gaby: Great'

### Debug when there is decorators: use functools.wraps decorator
Decorator process "hides" some of the meta data attached to the original (undecorated) function. 

In [164]:
def greet():
    '''Returns a friendly greeting'''
    return 'hello'

In [165]:
greet.__name__

'greet'

In [166]:
greet.__doc__

'Returns a friendly greeting'

In [167]:
decorated_greet= uppercase(greet)

In [168]:
decorated_greet.__name__

'wrapper'

In [169]:
decorated_greet.__doc__

In [170]:
# use functools
import functools

def uppercase(func):
    @functools.wraps(func)
    def wrapper():
        return func().upper()
    return wrapper
    

In [171]:
decorated_greet= uppercase(greet)

In [172]:
decorated_greet.__name__

'greet'

In [173]:
decorated_greet.__doc__

'Returns a friendly greeting'

## 3.4 Fun with *args and **kwargs

*args and **kwargs allow a function to accept _optional_ arguments, so you can create flexible APIs in your modules and classes. 

3 types of arguments:
- required arguments:
- *args: collect extra positional arguments as a tuple because the parameter name has a * prefix
- **kwargs: collect extra keyword arguments as a dictionary because the parameter name has a ** prefix.

NOTE: Actual syntax is just * and **, args and kwargs are the names by convention. 

In [195]:
# NOTE: all arguments before args are required

def foo(required, *args, **kwargs):
    print(required)
    if args:
        print(f'args are: {args}')
    if kwargs:
        print(f'kwargs are: {kwargs}')

In [196]:
foo('hi', 1)

hi
args are: (1,)


In [197]:
foo('hi', 3, 2.4)

hi
args are: (3, 2.4)


In [198]:
foo('hi',name='Gaby')

hi
kwargs are: {'name': 'Gaby'}


In [199]:
foo('hi', 3, 2.4, name='Gaby')

hi
args are: (3, 2.4)
kwargs are: {'name': 'Gaby'}


In [201]:
foo('hi', 3, 2.4, name='Gaby', anotherName = 'Toby')

hi
args are: (3, 2.4)
kwargs are: {'name': 'Gaby', 'anotherName': 'Toby'}


### Forwarding Optional or Keyword Arguments
Use * and ** as argument-unpacking operators. 

In [209]:
def foo(x, *args, **kwargs):
    kwargs['name'] = 'Alice'
    new_args = args + ('Bob', )
    # apply the arguments to another function:
    #bar(x, *new_args, **kwargs)

    return new_args

In [210]:
foo(1)

('Bob',)

In [213]:
# Can also be used in class
class Car:
    def __init__(self, color, mileage):
        self.color = color
        self.mileage = mileage


class BlueCar(Car):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.color = 'blue'

In [219]:
BlueCar('green', 2000).color

'blue'

In [220]:
BlueCar('green', 2000).mileage

2000

In [221]:
BlueCar('green').color

TypeError: Car.__init__() missing 1 required positional argument: 'mileage'

BlueCar cannot be defined without the mileage argument, though in the subclass, not all the arguments need to be specified thanks to *args and **kwargs. BUT, in this case we don't know what are the arguments to pass for BlueCar explicitly. This typically happens when we want to change external class that we don't have control. 

Another use case is writing wrapper functions such as decorators. 

## 3.5 Function Argument Unpacking

- "*": unpack sequences.
- "**": unpack dictionaries.

In [224]:
def print_vector(x, y, z):
    print('<%s, %s, %s>' % (x, y, z))
    

In [225]:
print_vector(1,2,3)

<1, 2, 3>


In [230]:
tuple_vec = (3.2, 232, 10)
list_vec  = [3.2, 232, 10]


In [231]:
print_vector(*tuple_vec)

<3.2, 232, 10>


BUT! in this case the function only accepts sequences with 3 elements. 

In [235]:
genexpr = (x*x for x in range(3))
print(genexpr)

<generator object <genexpr> at 0x106a7eab0>


In [236]:
print_vector(*genexpr)

<0, 1, 4>


In [238]:
dict_vec = {'a': 1, 'b': 2, 'c': 3}
print_vector(**dict_vec)

TypeError: print_vector() got an unexpected keyword argument 'a'

In [239]:
dict_vec = {'x': 1, 'y': 2, 'z': 3}
print_vector(**dict_vec)

<1, 2, 3>


## 3.6 Function that returns null

Python adds an implicit return null statement at end of each function. 

Author's rule of thumb: if a function doesn't have a return value (other languages would call this a _procedure_, example is a function with only print), then he'll leave out the return statement. 